# 01 — Environment & Intelligence
## Agentic X | Studio 67

This notebook prepares the simulated event environment, validates the scenario dataset, defines agent and environment characteristics, and executes the Creative LLM Agent.

The system handles event crises through four collaborating agents: Creative, Finance, Operations, and Safety.


Cell 2 — Connect to GitHub

In [1]:

from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )

os.chdir(ROOT)

print("Project directory:", Path.cwd())


Project directory: /content/AgenticX


Cell 3 — Install Requirements

In [2]:

import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt"
    ],
    check=True
)

print("Dependencies installed successfully")


Dependencies installed successfully


Cell 4 — Import Libraries

In [3]:

import pandas as pd
from IPython.display import display

from core.environment import (
    DEFAULT_DATA,
    ENVIRONMENT_DESCRIPTION,
    HIDDEN_OPTION_FIELDS,
    HIDDEN_CASE_FIELDS,
    load_dataset,
    get_case
)

from agents.creative import CreativeAgent

print("All imports completed")


All imports completed


Cell 5 — Load the Dataset


In [4]:

cases_df, options_df = load_dataset()

print("Total scenarios:", len(cases_df))
print("Total solution options:", len(options_df))
print("Unique cases:", cases_df["case_id"].nunique())

display(cases_df.head())
display(options_df.head())


Total scenarios: 150
Total solution options: 450
Unique cases: 150


,case_id,scenario,incident,budget_sar,time_left_min,severity,backup_available,crew_available,venue_access,notes,...,initial_budget_sar,committed_spend_sar,incident_type,required_quality_min,safety_policy,case_profile,crew_count_available,unplanned_charge_sar,projected_total_spend_sar,projected_overrun_sar
0,SCR-001,Screen Failure,Main LED screen failed during final rehearsal,8500,100,Critical,True,True,True,Synthetic test input; no precomputed agent dec...,...,8500,0,display_failure,60,Reject non-compliant options; escalate if none...,Baseline feasible,3,0,0,0
1,SCR-002,Screen Failure,LED wall lost signal before keynote,2400,60,High,False,True,True,Synthetic test input; no precomputed agent dec...,...,2400,0,display_failure,50,Reject non-compliant options; escalate if none...,Tight budget feasible,2,0,0,0
2,SCR-003,Screen Failure,Display controller failure before exhibition o...,7220,90,High,True,True,True,Synthetic test input; no precomputed agent dec...,...,7220,0,display_failure,50,Reject non-compliant options; escalate if none...,Time pressure feasible,2,0,0,0
3,SCR-004,Screen Failure,Screen power module stopped working,14000,150,Medium,False,True,True,Synthetic test input; no precomputed agent dec...,...,14000,0,display_failure,50,Reject non-compliant options; escalate if none...,Resource-constrained feasible,3,0,0,0
4,SCR-005,Screen Failure,Main stage screen shows intermittent blackout,11000,150,Critical,True,True,True,Synthetic test input; no precomputed agent dec...,...,11000,0,display_failure,55,Reject non-compliant options; escalate if none...,Multi-option feasible,1,0,0,0


,case_id,option_id,option_name,cost_sar,duration_min,safety_ok,quality_score,needs_backup,needs_crew,requires_venue_access,...,crew_required,risk_score_after_mitigation,safety_rejection_reason,safety_assessment_basis,gross_savings_sar,net_savings_sar,projected_total_after_sar,projected_overrun_after_sar,addresses_incident,resolution_explanation
0,SCR-001,SCR-001-O1,Use backup LED,1290,40,True,89,True,True,True,...,1,1,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...
1,SCR-001,SCR-001-O2,Rent replacement screen,5920,70,True,89,False,True,True,...,1,1,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...
2,SCR-001,SCR-001-O3,Continue with projection,1210,25,True,60,False,True,True,...,1,3,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...
3,SCR-002,SCR-002-O1,Use backup LED,1230,40,True,75,True,True,True,...,1,2,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...
4,SCR-002,SCR-002-O2,Rent replacement screen,6300,80,True,96,False,True,True,...,1,3,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...


Cell 6 — Validate the Dataset

In [5]:

assert len(cases_df) == 150
assert len(options_df) == 450

assert cases_df["case_id"].is_unique
assert options_df["option_id"].is_unique

assert not cases_df["case_id"].isna().any()
assert not options_df["case_id"].isna().any()

assert set(options_df["case_id"]).issubset(
    set(cases_df["case_id"])
)

options_per_case = options_df.groupby("case_id").size()

assert (options_per_case == 3).all()

print("Dataset validation passed")
print("\nScenario distribution:")
print(cases_df["scenario"].value_counts())


Dataset validation passed

Scenario distribution:
scenario
Screen Failure     50
Budget Overrun     50
Safety Conflict    50
Name: count, dtype: int64


Cell 7 — Define the Simulated Environment.

In [6]:

environment = {
    "name": "Studio 67",
    "domain": "Event Crisis Management",
    "agents": [
        "Creative",
        "Finance",
        "Operations",
        "Safety"
    ],
    "resources": [
        "Budget",
        "Time",
        "Crew",
        "Equipment",
        "Venue"
    ],
    "scenario_types": sorted(
        cases_df["scenario"].unique().tolist()
    )
}

for key, value in environment.items():
    print(f"{key}: {value}")


name: Studio 67
domain: Event Crisis Management
agents: ['Creative', 'Finance', 'Operations', 'Safety']
resources: ['Budget', 'Time', 'Crew', 'Equipment', 'Venue']
scenario_types: ['Budget Overrun', 'Safety Conflict', 'Screen Failure']


Cell 8 — Environment Classification

In [7]:

classification = {
    "Observability": (
        "Partially Observable",
        "Some operational information may be incomplete."
    ),
    "Determinism": (
        "Stochastic",
        "Real event disruptions involve uncertainty."
    ),
    "Dynamics": (
        "Dynamic",
        "Time and resource availability may change."
    ),
    "State Space": (
        "Mixed",
        "Options are discrete, while cost and time are numeric."
    )
}

classification_df = pd.DataFrame(
    [
        {
            "Property": key,
            "Classification": value[0],
            "Justification": value[1]
        }
        for key, value in classification.items()
    ]
)

display(classification_df)


,Property,Classification,Justification
0,Observability,Partially Observable,Some operational information may be incomplete.
1,Determinism,Stochastic,Real event disruptions involve uncertainty.
2,Dynamics,Dynamic,Time and resource availability may change.
3,State Space,Mixed,"Options are discrete, while cost and time are ..."


Cell 9 — Creative Agent PEAS

In [8]:

creative_peas = {
    "Performance Measure": (
        "Solution relevance, audience experience, "
        "and event quality"
    ),
    "Environment": (
        "Studio 67 event crisis simulation"
    ),
    "Actuators": (
        "Evaluate options, propose solutions, "
        "send recommendations"
    ),
    "Sensors": (
        "Incident details, solution descriptions, "
        "quality requirements, agent feedback"
    )
}

peas_df = pd.DataFrame(
    creative_peas.items(),
    columns=["PEAS Component", "Description"]
)

display(peas_df)

print("Proposed Agent Type: Deliberative")
print(
    "Justification: Creative reasons about alternatives "
    "before recommending a solution."
)


,PEAS Component,Description
0,Performance Measure,"Solution relevance, audience experience, and e..."
1,Environment,Studio 67 event crisis simulation
2,Actuators,"Evaluate options, propose solutions, send reco..."
3,Sensors,"Incident details, solution descriptions, quali..."


Proposed Agent Type: Deliberative
Justification: Creative reasons about alternatives before recommending a solution.


Cell 10 — Select a Test Scenario

In [9]:

CASE_ID = "SCR-001"

case, options = get_case(CASE_ID)

print("Case ID:", case["case_id"])
print("Scenario:", case["scenario"])
print("Incident:", case["incident"])
print("Available budget:", case["initial_budget_sar"])
print("Time remaining:", case["time_left_min"])

print("\nAvailable solutions:")

display(
    pd.DataFrame(options)[
        [
            "option_id",
            "option_name",
            "cost_sar",
            "duration_min",
            "quality_score"
        ]
    ]
)


Case ID: SCR-001
Scenario: Screen Failure
Incident: Main LED screen failed during final rehearsal
Available budget: 8500
Time remaining: 100

Available solutions:


,option_id,option_name,cost_sar,duration_min,quality_score
0,SCR-001-O1,Use backup LED,1290,40,89
1,SCR-001-O2,Rent replacement screen,5920,70,89
2,SCR-001-O3,Continue with projection,1210,25,60


Cell 11 — Prevent Evaluation Data Leakage

In [10]:

assert "case_profile" not in case

for option in options:
    assert "addresses_incident" not in option

    for field in HIDDEN_OPTION_FIELDS:
        assert field not in option

print("No evaluation ground-truth labels exposed")
print("Creative will assess the options independently")


No evaluation ground-truth labels exposed
Creative will assess the options independently


Cell 12 — Configure LLM Access

In [11]:

import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

os.environ.setdefault(
    "AGENTICX_MODEL",
    "gpt-4o-mini"
)

print("LLM configuration completed")


Enter your OpenAI API key: ··········
LLM configuration completed


Cell 13 — Initialize Creative Agent

In [12]:

creative_agent = CreativeAgent()

print("Agent initialized:", creative_agent.name)
print("Agent type: LLM-powered deliberative agent")


Agent initialized: Creative
Agent type: LLM-powered deliberative agent


Cell 14 — Evaluate the Three Solutions

In [13]:

creative_results = []

for option in options:

    print("Evaluating:", option["option_name"])

    evaluation = creative_agent.evaluate(
        case,
        option
    )

    creative_results.append(
        evaluation.model_dump()
    )

results_df = pd.DataFrame(creative_results)

display(results_df)


Evaluating: Use backup LED


OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************QOnQ. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}